# Notebook 5 - Feature engineering

Reads the three splits, writes `05_features_{train,val,test}.parquet`, `05_transformers.joblib`
and `05_feature_list.json`.

`val` and `test` are opened here, but only to apply. Every statistic - a median, a category list, a
late rate - is computed on train and then applied to the rest.

And every transform that learns something has to be saved: in production a single order arrives and
no median can be computed from it, so the pipeline has to load the fitted objects back.

## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import json
import joblib
from pathlib import Path
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import KFold

ARTIFACTS = Path("..") / "artifacts"

splits = {name: pd.read_parquet(ARTIFACTS / f"03_{name}.parquet")
          for name in ["train", "val", "test"]}

for name, df in splits.items():
    print(f"{name:6} {len(df):>7,} rows   {100*df['is_late'].mean():5.2f}% late")

TARGET = "is_late"
y_train = splits["train"][TARGET]

train   64,320 rows    7.95% late
val     13,547 rows    5.53% late
test    18,603 rows    3.61% late


## 1. The block list

Written in the code, with an `assert` that fails if one of these reaches
the final table.

In [2]:
# columns that materialise after the moment of purchase - using them means predicting the past
BLOCKED = [
    "order_approved_at",             # happens after the purchase
    "order_delivered_carrier_date",  # the actual handover to the carrier
    "order_delivered_customer_date", # the actual delivery - the answer itself
    "order_status",                  # the final status
    "days_late",                     # derived straight from the label
    "is_late", "is_late_ts", "is_late_cal",
]

# identifiers: not features, but not leakage either - used for joining and then left behind
IDENTIFIERS = ["order_id", "customer_id", "customer_unique_id", "main_seller_id",
               "customer_zip_code_prefix", "main_seller_zip"]

print(f"blocked     : {len(BLOCKED)} columns")
print(f"identifiers : {len(IDENTIFIERS)} columns")

blocked     : 8 columns
identifiers : 6 columns


### `shipping_limit_first`

The seller's deadline to hand the parcel over. Whether it is a plan set at purchase time
(legitimate) or a record of what happened (leakage) is measurable.

In [3]:
tr = splits["train"]
limit_days   = (tr["shipping_limit_first"] - tr["order_purchase_timestamp"]).dt.total_seconds()/86400
carrier_days = (tr["order_delivered_carrier_date"] - tr["order_purchase_timestamp"]).dt.total_seconds()/86400

print(f"smallest gap from the purchase : {limit_days.min():.2f} days  (no negative values)")
print(f"median                         : {limit_days.median():.2f} days")
print(f"correlation with the actual carrier handover: {limit_days.corr(carrier_days, method='spearman'):.2f}")

smallest gap from the purchase : 3.96 days  (no negative values)
median                         : 6.01 days
correlation with the actual carrier handover: 0.25


Legitimate. No value precedes the purchase, the ~4 day minimum looks like a fixed commercial rule
applied at order creation, and its correlation with the actual carrier handover is 0.25 - a record
of the event would sit near 1.

It goes in as `shipping_limit_days`. Weak signal (AUC ~0.49), no cost.

## 2. Derived features

These are pure functions of a single row and learn nothing from the rest of the data, so one
function serves all three splits. Any difference between what is applied to train and what is
applied to test would be a silent bug.

In [4]:
EARTH_RADIUS_KM = 6371.0

def haversine_km(lat1, lon1, lat2, lon2):
    '''Distance on the surface of a sphere between two points given in degrees.'''
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = (np.sin((lat2 - lat1)/2)**2
         + np.cos(lat1)*np.cos(lat2)*np.sin((lon2 - lon1)/2)**2)
    return 2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(a))


def build_derived(df):
    '''
    Every feature computed from the row itself, with no knowledge of the rest of the data.
    Applied literally unchanged to train, val and test.
    '''
    f = pd.DataFrame(index=df.index)
    ts = df["order_purchase_timestamp"]

    # --- the promise and time (all known at the moment of purchase) ---
    f["promised_days"] = (df["order_estimated_delivery_date"] - ts).dt.total_seconds() / 86400
    f["shipping_limit_days"] = (df["shipping_limit_first"] - ts).dt.total_seconds() / 86400
    f["purchase_dow"]  = ts.dt.dayofweek
    f["purchase_hour"] = ts.dt.hour

    # --- geography ---
    f["distance_km"] = haversine_km(df["customer_lat"], df["customer_lng"],
                                    df["seller_lat"],   df["seller_lng"])
    # the missingness is itself a signal (verdict 4 in notebook 4) - built before any filling
    f["customer_geo_missing"] = df["customer_lat"].isna().astype(int)
    f["seller_geo_missing"]   = df["seller_lat"].isna().astype(int)
    f["same_state"] = (df["customer_state"] == df["main_seller_state"]).astype(int)

    # --- the strongest feature the EDA found: the promise relative to the distance ---
    hundreds_of_km = (f["distance_km"] / 100).replace(0, np.nan)
    f["promise_per_100km"] = f["promised_days"] / hundreds_of_km

    # --- money and freight ---
    f["freight_total"] = df["freight_total"]
    f["freight_max"]   = df["freight_max"]
    f["freight_ratio"] = df["freight_total"] / df["items_price_total"].replace(0, np.nan)
    f["items_price_total"] = df["items_price_total"]
    f["items_price_max"]   = df["items_price_max"]
    f["payment_total"]     = df["payment_total"]
    f["installments_max"]  = df["installments_max"]
    f["n_payments"]        = df["n_payments"]

    # --- basket and product (weak signal, zero cost) ---
    f["n_items"]        = df["n_items"]
    f["n_products"]     = df["n_products"]
    f["n_sellers"]      = df["n_sellers"]
    f["n_categories"]   = df["n_categories"]
    f["weight_g_total"] = df["weight_g_total"]
    f["photos_avg"]     = df["photos_avg"]

    return f


derived = {name: build_derived(df) for name, df in splits.items()}
print(f"derived features: {derived['train'].shape[1]} columns")
derived["train"].head(3)

derived features: 23 columns


,promised_days,shipping_limit_days,purchase_dow,purchase_hour,distance_km,customer_geo_missing,seller_geo_missing,same_state,promise_per_100km,freight_total,...,items_price_max,payment_total,installments_max,n_payments,n_items,n_products,n_sellers,n_categories,weight_g_total,photos_avg
0,15.544063,4.007431,0,10,18.681711,0,0,1,83.204704,8.72,...,29.99,38.71,1.0,3.0,1.0,1.0,1.0,1.0,500.0,4.0
1,26.188819,5.012419,5,19,1821.802656,0,0,0,1.437522,27.20,...,45.00,72.20,1.0,1.0,1.0,1.0,1.0,1.0,450.0,3.0
2,21.451042,6.008113,1,13,322.312700,0,0,0,6.655351,15.17,...,59.99,75.16,3.0,1.0,1.0,1.0,1.0,1.0,50.0,1.0


`month`, `year` and any absolute time marker are left out. Notebook 4 found the month strong
(0.75% to 19%), but training covers a year and a half, so "March" in this data is March 2018.

`volume_cm3_total` is out too: 0.78 correlated with weight.

## 3. Computed versus fitted

`distance_km` from two coordinates is a pure function - the code is the definition. Filling a gap
with the train median is a fitted transform - the median is a learned number and has to be saved.

All the danger is in the second kind. The fitted objects are collected in one `transformers` dict.

In [5]:
transformers = {}

## 4. Grouping rare categories

28 product categories hold fewer than 100 orders and cover 1.5% of the data; left alone they
produce near-empty categories that carry noise.

The kept list comes from train and is applied unchanged. A category that appears only in test
becomes `"rare"`, which is the right behaviour in production.

In [6]:
MIN_CATEGORY_COUNT = 100

counts = splits["train"]["main_category"].value_counts()
kept_categories = sorted(counts[counts >= MIN_CATEGORY_COUNT].index.tolist())

transformers["kept_categories"] = kept_categories

def group_rare(series, kept):
    '''Categories outside the list -> "rare", and a gap -> "unknown".'''
    return series.where(series.isin(kept), "rare").fillna("unknown")

print(f"categories in train            : {len(counts)}")
print(f"kept (>= {MIN_CATEGORY_COUNT} orders)              : {len(kept_categories)}")
print(f"grouped into 'rare'            : {len(counts) - len(kept_categories)}")

categories in train            : 72
kept (>= 100 orders)              : 44
grouped into 'rare'            : 28


## 5. Target encoding with smoothing

`customer_state`, `main_seller_state` and `main_category` carry real signal (4.7% to 19.4% by
state), so each category is replaced by its historical late rate. Two problems have to be solved
first.

Small categories: a state with 12 orders and 2 late gives 16.7%, which is noise. Smoothing pulls
the estimate towards the overall mean with a strength inversely proportional to the category size.

$$\text{encoded} = \frac{n \cdot \text{mean}_{\text{category}} + m \cdot \text{prior}}{n + m}$$

Leakage inside train: encoding train rows with a mean computed from train lets every row see its
own label. Out-of-fold encoding fixes it - five folds, each encoded from the other four.

`val` and `test` use the statistics of the whole train set, as production would.

In [7]:
SMOOTHING = 50   # pull strength towards the overall mean: equivalent to "50 imaginary orders at the overall rate"

def fit_target_encoder(x, y, smoothing=SMOOTHING):
    '''Learns a category -> smoothed late rate map. Returns plain data (saveable).'''
    prior = float(y.mean())
    stats = y.groupby(x).agg(["count", "mean"])
    smoothed = ((stats["count"]*stats["mean"] + smoothing*prior)
                / (stats["count"] + smoothing))
    return {"mapping": smoothed.to_dict(), "prior": prior, "smoothing": smoothing}

def apply_target_encoder(x, encoder):
    '''An unknown category -> the overall mean. This is what will happen in production with a new category.'''
    return x.map(encoder["mapping"]).fillna(encoder["prior"]).astype(float)

In [8]:
TARGET_ENCODE = ["customer_state", "main_seller_state", "main_category"]
N_FOLDS = 5

# prepare the categorical columns in all three splits (with rare grouping for the categories)
cats = {}
for name, df in splits.items():
    cats[name] = pd.DataFrame({
        "customer_state":    df["customer_state"].fillna("unknown"),
        "main_seller_state": df["main_seller_state"].fillna("unknown"),
        "main_category":     group_rare(df["main_category"], kept_categories),
    })

transformers["target_encoders"] = {}

for col in TARGET_ENCODE:
    x_tr = cats["train"][col]

    # (a) the final map from the whole train set - for val, test and production
    encoder = fit_target_encoder(x_tr, y_train)
    transformers["target_encoders"][col] = encoder

    # (b) inside train: out-of-fold encoding so that no row sees its own label
    oof = pd.Series(np.nan, index=x_tr.index, dtype=float)
    for fold_train, fold_valid in KFold(N_FOLDS, shuffle=True, random_state=42).split(x_tr):
        fold_encoder = fit_target_encoder(x_tr.iloc[fold_train], y_train.iloc[fold_train])
        oof.iloc[fold_valid] = apply_target_encoder(x_tr.iloc[fold_valid], fold_encoder).values

    derived["train"][f"{col}_te"] = oof
    for name in ["val", "test"]:
        derived[name][f"{col}_te"] = apply_target_encoder(cats[name][col], encoder)

    print(f"{col:20} {len(encoder['mapping']):>3} categories   "
          f"range {min(encoder['mapping'].values()):.3f} -> {max(encoder['mapping'].values()):.3f}")

customer_state        27 categories   range 0.048 -> 0.224


main_seller_state     22 categories   range 0.038 -> 0.207


main_category         45 categories   range 0.042 -> 0.145


In [9]:
# check: did the smoothing do what we want? we compare a large state with a small one
enc = transformers["target_encoders"]["customer_state"]
raw = y_train.groupby(cats["train"]["customer_state"]).agg(["count", "mean"])
raw["smoothed"] = pd.Series(enc["mapping"])
raw.columns = ["orders", "raw rate", "after smoothing"]
raw.sort_values("orders").head(4).round(4)

,orders,raw rate,after smoothing
customer_state,,,
RR,28,0.1429,0.1022
AP,46,0.0435,0.0622
AC,64,0.0312,0.0524
AM,101,0.0396,0.0528


## 6. Payment type

Four values only, so one-hot is simpler than target encoding here. `handle_unknown="ignore"` means
an unseen payment type becomes zeros instead of raising.

In [10]:
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.int8)
ohe.fit(splits["train"][["main_payment_type"]].fillna("unknown"))

transformers["onehot_payment"] = ohe
ohe_names = [f"pay_{v}" for v in ohe.categories_[0]]

for name in splits:
    encoded = ohe.transform(splits[name][["main_payment_type"]].fillna("unknown"))
    for i, col in enumerate(ohe_names):
        derived[name][col] = encoded[:, i]

print("payment columns:", ohe_names)

payment columns: ['pay_boleto', 'pay_credit_card', 'pay_debit_card', 'pay_unknown', 'pay_voucher']


## 7. Filling the gaps

The median comes from train and is saved - including when filling test, since a test median would
put a description of the future into the data. The missingness indicators were built in section 2,
before the filling, so the information survives.

In [11]:
FEATURE_COLUMNS = sorted(derived["train"].columns.tolist())

missing_before = derived["train"][FEATURE_COLUMNS].isna().sum()
print("columns with gaps before filling:")
print(missing_before[missing_before > 0].to_string())

columns with gaps before filling:


distance_km           330
installments_max        1
n_payments              1
payment_total           1
photos_avg           1146
promise_per_100km     343
weight_g_total         16


In [12]:
imputer = SimpleImputer(strategy="median")
imputer.fit(derived["train"][FEATURE_COLUMNS])

transformers["imputer"] = imputer
transformers["feature_columns"] = FEATURE_COLUMNS

features = {}
for name in splits:
    filled = imputer.transform(derived[name][FEATURE_COLUMNS])
    features[name] = pd.DataFrame(filled, columns=FEATURE_COLUMNS,
                                  index=derived[name].index)

for name, f in features.items():
    print(f"{name:6} {f.shape[0]:>7,} x {f.shape[1]}   NaN remaining: {int(f.isna().sum().sum())}")

train   64,320 x 31   NaN remaining: 0
val     13,547 x 31   NaN remaining: 0
test    18,603 x 31   NaN remaining: 0


## 8. Scaling

Trees do not need it, logistic regression does. The scaler is fitted and saved but not applied to
the tables, which stay readable; notebook 6 applies it to the linear model only.

In [13]:
scaler = StandardScaler()
scaler.fit(features["train"])
transformers["scaler"] = scaler

print("scaler fitted and saved (not applied to the tables).")
print(f"example - mean and spread of distance_km in train: "
      f"{scaler.mean_[FEATURE_COLUMNS.index('distance_km')]:.0f} km, "
      f"sigma = {scaler.scale_[FEATURE_COLUMNS.index('distance_km')]:.0f}")

scaler fitted and saved (not applied to the tables).
example - mean and spread of distance_km in train: 616 km, sigma = 595


## 9. Checks

In [14]:
# 1) no blocked column sneaked into the features
leaked = [col for col in FEATURE_COLUMNS if col in BLOCKED or col in IDENTIFIERS]
assert not leaked, f"leaking columns in the features: {leaked}"

# 2) the three splits carry the same columns in the same order
assert list(features["train"].columns) == list(features["val"].columns) == list(features["test"].columns), \
    "the columns do not match across the splits"

# 3) no missing and no infinite values
for name, f in features.items():
    assert f.isna().sum().sum() == 0, f"NaN remaining in {name}"
    assert np.isfinite(f.to_numpy()).all(), f"infinite values in {name}"

# 4) the row counts did not change
for name in splits:
    assert len(features[name]) == len(splits[name]), f"the row count changed in {name}"

print(f"OK  {len(FEATURE_COLUMNS)} features, no leakage, no gaps, identical structure across splits")

OK  31 features, no leakage, no gaps, identical structure across splits


In [15]:
# a final sanity check: do the features look statistically similar between train and test?
# large gaps here are not necessarily a bug - but they are a drift signal worth knowing.
compare = pd.DataFrame({
    "train median": features["train"].median(),
    "test median":  features["test"].median(),
})
compare["change %"] = (100*(compare["test median"] - compare["train median"])
                       / compare["train median"].replace(0, np.nan)).round(1)
compare.reindex(compare["change %"].abs().sort_values(ascending=False).index).head(8)

,train median,test median,change %
installments_max,2.000000,1.000000,-50.0
weight_g_total,800.000000,650.000000,-18.8
shipping_limit_days,6.013843,5.007014,-16.7
promised_days,23.699433,20.028403,-15.5
freight_max,16.110000,18.330000,13.8
distance_km,451.600133,400.108207,-11.4
freight_total,16.740000,18.630000,11.3
customer_state_te,0.052495,0.047764,-9.0


## 10. Saving

Three outputs: the feature tables for notebook 6, the fitted objects for running the pipeline on
new data, and the feature list as the contract on column names and order.

In [16]:
# 1) the feature tables + the label + the order id (for tracing, not for training)
for name in splits:
    out = features[name].copy()
    out.insert(0, "order_id", splits[name]["order_id"].values)
    out[TARGET] = splits[name][TARGET].values
    path = ARTIFACTS / f"05_features_{name}.parquet"
    out.to_parquet(path, index=False)
    print(f"{path.name:<28} {len(out):>7,} x {out.shape[1]}")

05_features_train.parquet     64,320 x 33


05_features_val.parquet       13,547 x 33


05_features_test.parquet      18,603 x 33


In [17]:
# 2) the fitted objects
transformers["metadata"] = {
    "trained_on": "03_train.parquet",
    "n_train_rows": int(len(splits["train"])),
    "train_base_rate": float(y_train.mean()),
    "target": TARGET,
    "smoothing": SMOOTHING,
    "n_folds_oof": N_FOLDS,
    "min_category_count": MIN_CATEGORY_COUNT,
}

path = ARTIFACTS / "05_transformers.joblib"
joblib.dump(transformers, path)
print(f"{path.name}  ({path.stat().st_size/1024:.0f} KB)")
print("contents:", list(transformers.keys()))

05_transformers.joblib  (8 KB)
contents: ['kept_categories', 'target_encoders', 'onehot_payment', 'imputer', 'feature_columns', 'scaler', 'metadata']


In [18]:
# 3) the feature list - the contract between this notebook and any later production code
feature_list = {
    "target": TARGET,
    "n_features": len(FEATURE_COLUMNS),
    "features": FEATURE_COLUMNS,
    "blocked_columns": BLOCKED,
    "needs_scaling_for": "linear models only - use transformers['scaler']",
    "metadata": transformers["metadata"],
}

path = ARTIFACTS / "05_feature_list.json"
path.write_text(json.dumps(feature_list, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"{path.name}  - {len(FEATURE_COLUMNS)} features")

05_feature_list.json  - 31 features


### Round-trip check

The objects are loaded back from disk and used to rebuild one order from scratch, then compared
against what the notebook produced.

In [19]:
loaded = joblib.load(ARTIFACTS / "05_transformers.joblib")

# simulate production: one order from test, passed through the loaded objects alone
one_order = splits["test"].iloc[[0]]

rebuilt = build_derived(one_order)
for col in TARGET_ENCODE:
    source = (group_rare(one_order["main_category"], loaded["kept_categories"])
              if col == "main_category" else one_order[col].fillna("unknown"))
    rebuilt[f"{col}_te"] = apply_target_encoder(source, loaded["target_encoders"][col])

encoded = loaded["onehot_payment"].transform(one_order[["main_payment_type"]].fillna("unknown"))
for i, col in enumerate([f"pay_{v}" for v in loaded["onehot_payment"].categories_[0]]):
    rebuilt[col] = encoded[:, i]

rebuilt = pd.DataFrame(loaded["imputer"].transform(rebuilt[loaded["feature_columns"]]),
                       columns=loaded["feature_columns"])

original = features["test"].iloc[[0]].reset_index(drop=True)
assert np.allclose(rebuilt.to_numpy(), original.to_numpy()), "the output does not match!"
print("OK  rebuilding from the loaded objects matches the notebook output exactly")
print("    -> the pipeline is genuinely reproducible, not just claimed to be")

OK  rebuilding from the loaded objects matches the notebook output exactly
    -> the pipeline is genuinely reproducible, not just claimed to be


---

**Result:** 3 feature tables with 31 features, `05_transformers.joblib`, `05_feature_list.json`.

| Saved object | What it learned |
|---|---|
| `kept_categories` | which product categories are not rare |
| `target_encoders` | the late rate per state and category |
| `imputer` | the median of every feature |
| `onehot_payment` | the payment values and their column order |
| `scaler` | mean and deviation per feature |

Refitting any of these in production shifts the meaning of the columns underneath a model that was
trained on the old ones.

Decisions: out-of-fold target encoding inside train, smoothing at 50, missingness indicators built
before the filling, train medians even when filling test, the scaler saved but not applied, and
`month`/`year` left out despite their signal.